In [34]:
import pandas as pd

In [23]:
df_calls = pd.read_csv("../../dataset/03_EDA/cc_calls/cc_calls_eda.csv")

In [24]:
# Hypothesis
# H0: Contact_ID is independent of churn
# H1: Contact_ID is associated with churn

In [25]:
import pandas as pd
from scipy.stats import chi2_contingency

data = df_calls[['Contact_ID', 'target']].dropna()

table = pd.crosstab(data['Contact_ID'], data['target'])

chi2, p, dof, exp = chi2_contingency(table)

print("P-value:", p)

if p > 0.05:
    print("❌ Fail to Reject H0 (as expected)")
else:
    print("⚠️ Unexpected result (check data leakage)")

P-value: 0.09009213349806199
❌ Fail to Reject H0 (as expected)


In [26]:
# Hypothesis
# H0: External consultant mention is independent of churn
# H1: It affects churn

In [27]:
data = df_calls[['cc_external_consultant', 'target']].dropna()

table = pd.crosstab(data['cc_external_consultant'], data['target'])

from scipy.stats import chi2_contingency

chi2, p, _, _ = chi2_contingency(table)

print("P-value:", p)

if p > 0.05:
    print("❌ Fail to Reject H0 (likely)")
else:
    print("⚠️ May have niche signal")

P-value: 0.6466834619297253
❌ Fail to Reject H0 (likely)


In [28]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np

# Clean data
data = df_calls[['days_bucket', 'target']].dropna()

# Contingency table
table = pd.crosstab(data['days_bucket'], data['target'])

# Chi-square test
chi2, p, dof, expected = chi2_contingency(table)

# Effect size (IMPORTANT)
n = table.sum().sum()
r, k = table.shape
cramers_v = np.sqrt(chi2 / (n * (min(r, k) - 1)))

print("📊 Chi-Square Test: days_bucket vs Churn")
print("-" * 50)
print("Chi2:", chi2)
print("P-value:", p)
print("Cramér's V:", cramers_v)
print("\nContingency Table:\n", table)

# Decision logic (IMPORTANT FIX)
alpha = 0.05

if p < alpha and cramers_v > 0.05:
    print("\n✅ Reject H0: days_bucket has meaningful effect on churn")
elif p < alpha and cramers_v <= 0.05:
    print("\n⚠️ Statistically significant but weak effect (likely not useful)")
else:
    print("\n❌ Fail to Reject H0: days_bucket not related to churn")

print("\nExpected Frequencies:\n", expected)

📊 Chi-Square Test: days_bucket vs Churn
--------------------------------------------------
Chi2: 4.9474108451131
P-value: 0.17568873226619924
Cramér's V: 0.06954263968405214

Contingency Table:
 target         0   1
days_bucket         
14-21d       267  12
22-28d       208  10
29-35d       236   3
36-45d       277  10

❌ Fail to Reject H0: days_bucket not related to churn

Expected Frequencies:
 [[269.45454545   9.54545455]
 [210.54154448   7.45845552]
 [230.8230694    8.1769306 ]
 [277.18084066   9.81915934]]


In [29]:
print(df_calls['days_bucket'].value_counts())

days_bucket
36-45d    287
14-21d    279
29-35d    239
22-28d    218
Name: count, dtype: int64


In [35]:
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency

alpha = 0.05

fail_h0 = []
reject_h0 = []

num_cols = df_calls.select_dtypes(include=['int64','float64']).columns.drop('target', errors='ignore')
cat_cols = df_calls.select_dtypes(include=['object','category','bool']).columns

print("📊 FINDING FEATURES THAT FAIL TO REJECT H0")
print("="*60)

# ---------------- NUMERICAL FEATURES ----------------
for col in num_cols:
    data = df_calls[[col, 'target']].dropna()

    if data['target'].nunique() < 2:
        continue

    g1 = data[data['target'] == 1][col]
    g0 = data[data['target'] == 0][col]

    if len(g1) < 10 or len(g0) < 10:
        continue  # avoid tiny sample bias

    stat, p = mannwhitneyu(g1, g0, alternative='two-sided')

    print(f"[NUM] {col} -> p={p:.4f}")

    if p > alpha:
        fail_h0.append(col)
    else:
        reject_h0.append(col)

# ---------------- CATEGORICAL FEATURES ----------------
for col in cat_cols:
    data = df_calls[[col, 'target']].dropna()

    if data[col].nunique() < 2:
        continue

    table = pd.crosstab(data[col], data['target'])

    if table.shape[0] < 2:
        continue

    chi2, p, _, _ = chi2_contingency(table)

    print(f"[CAT] {col} -> p={p:.4f}")

    if p > alpha:
        fail_h0.append(col)
    else:
        reject_h0.append(col)

# ---------------- FINAL SUMMARY ----------------
print("\n" + "="*60)
print("❌ FEATURES THAT FAIL TO REJECT H0 (WEAK / NON-DRIVERS)")
print(fail_h0)

print("\n✅ FEATURES THAT REJECT H0 (STRONG DRIVERS)")
print(reject_h0)

print("\nTOTAL FAIL H0:", len(fail_h0))
print("TOTAL REJECT H0:", len(reject_h0))
print("="*60)

📊 FINDING FEATURES THAT FAIL TO REJECT H0
[NUM] Contact_ID -> p=0.0100
[NUM] cc_contractor_sentiment_start_score -> p=0.2970
[NUM] cc_contractor_sentiment_end_score -> p=0.0637
[NUM] cc_contractor_sentiment_overall_score -> p=0.2363
[NUM] Analysed_Call -> p=1.0000
[NUM] Call_Year -> p=0.1729
[NUM] Days_Before_Renewal -> p=0.2914
[NUM] sentiment_change -> p=0.3529
[NUM] negative_sentiment_flag -> p=0.7082
[CAT] Call_Date -> p=0.0025
[CAT] Direction -> p=0.9401
[CAT] cc_care_package -> p=0.2947
[CAT] cc_care_package_discussed -> p=0.7790
[CAT] cc_urgency_getting_on_site -> p=0.4525
[CAT] cc_external_consultant -> p=0.6467
[CAT] cc_agent_cross_sell_attempt -> p=1.0000
[CAT] cc_customer_issues_concerns -> p=0.7034
[CAT] cc_business_struggles_financial_hardship -> p=0.5728
[CAT] cc_call_initiated_by -> p=0.1412
[CAT] cc_questionnaire_completion -> p=0.2745
[CAT] cc_chasing_response -> p=0.9179
[CAT] cc_issues_within_questionnaire -> p=0.6226
[CAT] cc_login_issues -> p=0.2843
[CAT] cc_platfo

In [31]:
# df_calls['pricing_issue_flag'] = df_calls['cc_pricing_sentiment_impact'].replace({
#     'no': 0,
#     'unknown': 0,
#     'yes': 1
# })

In [32]:
# import pandas as pd
# from scipy.stats import chi2_contingency
# import numpy as np

# data = df_calls[['pricing_issue_flag', 'target']].dropna()

# table = pd.crosstab(data['pricing_issue_flag'], data['target'])

# chi2, p, _, _ = chi2_contingency(table)

# n = table.sum().sum()
# r, k = table.shape
# cramers_v = np.sqrt(chi2 / (n * (min(r,k)-1)))

# print("P-value:", p)
# print("Cramér's V:", cramers_v)
# print(table)

In [33]:
# import pandas as pd
# from scipy.stats import chi2_contingency
# import numpy as np

# # Clean data
# data = df_calls[['cc_pricing_sentiment_impact', 'target']].dropna()

# # Contingency table
# table = pd.crosstab(data['cc_pricing_sentiment_impact'], data['target'])

# # Chi-square test
# chi2, p, dof, expected = chi2_contingency(table)

# # Effect size (Cramér's V)
# n = table.sum().sum()
# r, k = table.shape
# cramers_v = np.sqrt(chi2 / (n * (min(r, k) - 1)))

# print("📊 Chi-Square Test: cc_pricing_sentiment_impact vs Churn")
# print("-" * 60)
# print("Chi2:", chi2)
# print("P-value:", p)
# print("Cramér's V:", cramers_v)

# print("\nContingency Table:\n", table)

# # Decision rule
# alpha = 0.05

# if p < alpha and cramers_v > 0.05:
#     print("\n✅ Reject H0: Pricing sentiment impact influences churn")
# elif p < alpha and cramers_v <= 0.05:
#     print("\n⚠️ Statistically significant but weak effect (not useful)")
# else:
#     print("\n❌ Fail to Reject H0: No relationship with churn")

# print("\nExpected Frequencies:\n", expected)

In [ ]:
# 🔬 Hypotheses
# H0: cc_contractor_suggest_leave has no relationship with churn
# H1: cc_contractor_suggest_leave is associated with churn

In [37]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np

# Clean data
data = df_calls[['cc_contractor_suggest_leave', 'target']].dropna()

# Contingency table
table = pd.crosstab(data['cc_contractor_suggest_leave'], data['target'])

# Chi-square test
chi2, p, dof, expected = chi2_contingency(table)

# Effect size (Cramér's V)
n = table.sum().sum()
r, k = table.shape
cramers_v = np.sqrt(chi2 / (n * (min(r, k) - 1)))

print("📊 Chi-Square Test: cc_contractor_suggest_leave vs Churn")
print("-" * 60)
print("Chi2:", chi2)
print("P-value:", p)
print("Cramér's V:", cramers_v)

print("\nContingency Table:\n", table)

# Decision logic
alpha = 0.05

if p < alpha and cramers_v > 0.05:
    print("\n✅ Reject H0: Suggesting leave is strongly associated with churn")
elif p < alpha and cramers_v <= 0.05:
    print("\n⚠️ Statistically significant but weak effect")
else:
    print("\n❌ Fail to Reject H0: No meaningful relationship with churn")

print("\nExpected Frequencies:\n", expected)

📊 Chi-Square Test: cc_contractor_suggest_leave vs Churn
------------------------------------------------------------
Chi2: 78.38029584628569
P-value: 3.8375550335931823e-16
Cramér's V: 0.2772064705238319

Contingency Table:
 target                                                0   1
cc_contractor_suggest_leave                                
 explained the necessary steps                        1   0
No                                                  964  26
The Contractor did not respond to the Agent's a...    2   0
The Contractor was appreciative of the Agent's ...    1   0
Yes                                                  17   9

✅ Reject H0: Suggesting leave is strongly associated with churn

Expected Frequencies:
 [[9.65686275e-01 3.43137255e-02]
 [9.56029412e+02 3.39705882e+01]
 [1.93137255e+00 6.86274510e-02]
 [9.65686275e-01 3.43137255e-02]
 [2.51078431e+01 8.92156863e-01]]


# Due to class imbalance and sparcity i get wrong test inmany cases 